# 00 Prepare Data

Objective: load the NICE/PROMISE-derived requirements dataset, create a transparent seed review table, and keep exactly 120 accepted seed capabilities for the main experiment.

The automatic filter and capability cleanup are intentionally conservative. Manual review remains part of the protocol, but `capability_text_final` is pre-filled with a cleaned suggestion so most rows should only need inspection rather than hand rewriting.


In [ ]:
from pathlib import Path
import os
import sys
import importlib

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
while not (PROJECT_ROOT / "AGENTS.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import eval_utils as eu
eu = importlib.reload(eu)

CONFIG_PATH = PROJECT_ROOT / "config.json"
if not CONFIG_PATH.exists():
    CONFIG_PATH = PROJECT_ROOT / "config.example.json"
CONFIG = eu.load_config(CONFIG_PATH)
eu.ensure_project_dirs(PROJECT_ROOT)
BENCHMARK_VARIANT = os.getenv("BENCHMARK_VARIANT", "must").strip().lower()
VARIANT_SUFFIX = eu.variant_suffix(BENCHMARK_VARIANT)

PROJECT_ROOT, CONFIG_PATH, BENCHMARK_VARIANT


## Load or Download NICE


In [ ]:
nice_path = PROJECT_ROOT / CONFIG["datasets"]["nice_local_path"]
nice_url = CONFIG["datasets"]["nice_url"]

if not nice_path.exists():
    print(f"NICE CSV not found at {nice_path}. Downloading from Zenodo...")
    eu.download_file(nice_url, nice_path, timeout_s=CONFIG["llm"]["timeout_s"])
else:
    print(f"Using existing NICE CSV: {nice_path}")

rows = eu.read_csv_rows(nice_path)
text_column = eu.find_requirement_text_column(rows)
print(f"Loaded {len(rows)} rows. Requirement text column: {text_column}")
print(rows[0][text_column][:240])


## Build Review Table


In [ ]:
target_count = int(CONFIG["project"]["target_seed_count"])
candidates = eu.make_seed_candidates(rows, target_count=target_count)

review_fields = [
    "seed_id",
    "source_dataset",
    "original_requirement",
    "capability_text_auto",
    "auto_include",
    "auto_exclusion_reason",
    "include",
    "exclusion_reason",
    "capability_text_final",
]
review_path = PROJECT_ROOT / "data/processed/seeds_review.csv"
auto_candidates_path = PROJECT_ROOT / "data/processed/seeds_review_candidates_auto.csv"

if review_path.exists():
    eu.write_csv_rows(auto_candidates_path, candidates, fieldnames=review_fields)
    print(f"Existing review table found: {review_path}")
    print("Did not overwrite reviewed/manual capability edits.")
    print(f"Wrote fresh automatic candidates for comparison: {auto_candidates_path}")
else:
    eu.write_csv_rows(review_path, candidates, fieldnames=review_fields)
    print(f"Wrote new review table: {review_path}")

auto_ok = sum(1 for row in candidates if row["auto_include"] == "yes")
selected = eu.load_reviewed_seeds(review_path, target_count=target_count, strict=False)
print(f"Automatic candidates passing filters: {auto_ok}")
print(f"Currently included seeds: {len(selected)} / {target_count}")
print("Inspect capability_text_final for included rows; edit only unclear or awkward suggestions.")


## Optionally Refresh Existing Capability Suggestions


In [ ]:
# Manual review edits are preserved by default.
# Set RUN_REFRESH = True only if you want to refresh unedited capability suggestions.
RUN_REFRESH = False
review_path = PROJECT_ROOT / "data/processed/seeds_review.csv"

if RUN_REFRESH:
    refreshed_count = eu.refresh_capability_suggestions_file(review_path)
    print(f"Refreshed capability_text_final suggestions: {refreshed_count}")
else:
    print("Skipped refresh; reviewed/manual capability edits were left untouched.")


## Inspect Included Capability Suggestions


In [ ]:
import pandas as pd

review_df = pd.read_csv(review_path, dtype=str, keep_default_na=False)
included = review_df[review_df["include"] == "yes"].copy()
suspicious = included[
    included["capability_text_final"].str.contains(r"\b(shall|must|should|may|system|product|application)\b", case=False, regex=True)
    | included["capability_text_final"].str.contains(r"[.;:]", regex=True)
]
print(f"Included rows: {len(included)}")
print(f"Rows worth closer manual review: {len(suspicious)}")
suspicious[["seed_id", "original_requirement", "capability_text_final"]].head(20)


## Show Full Included Capability Review Table


In [ ]:
capability_review = eu.included_capability_review_frame(review_path)
export_paths = eu.write_included_capability_review(review_path, PROJECT_ROOT / "outputs")

print(f"Included rows: {len(capability_review)}")
print(f"Wrote Markdown review table: {export_paths['markdown']}")
print(f"Wrote CSV review table: {export_paths['csv']}")

with pd.option_context("display.max_rows", None, "display.max_colwidth", 180, "display.width", 240):
    display(capability_review)


## Validate Reviewed Seeds


In [ ]:
selected = eu.load_reviewed_seeds(review_path, target_count=target_count, strict=False)
selected_path = PROJECT_ROOT / "data/processed/seeds_selected.csv"

if len(selected) == target_count:
    eu.write_csv_rows(selected_path, selected)
    print(f"OK: exactly {target_count} included seeds.")
    print(f"Wrote selected seeds: {selected_path}")
else:
    print(f"Review needed: found {len(selected)} included seeds, expected {target_count}.")
    print("Edit data/processed/seeds_review.csv, then rerun this cell.")

selected[:3]
